# ChessLab Chess Game Parsing

This notebook explores how ChessLab parses PGN data from Chess.com.

## Goals

1. Download PGN data from Chess.com
2. Parse a chess game using `python-chess`
3. Inspect game headers
4. Inspect the moves in the game
5. Understand the structure of a parsed chess game

The production parsing code lives in: `src/chesslab/chess/parser.py`

In [3]:
# import download and parse functions
from chesslab.api.chesscom import download_monthly_games

from chesslab.chess.parser import (
    parse_game, 
    parse_games,
    )

## 1. Select a player and month

For this exploration, we will download games from a single month.

In [4]:
# again, we'll select my own data from this month (Sept 2026)
username = "tjmoonz98"
year = 2026
month = 9

print(f"Downloading games for {username} — {year}-{month:02d}")

## 2. Download PGN data

The Chess.com API returns the games for the selected month as raw PGN text.

In [5]:
# define pgn object and print some character number stats
pgn = download_monthly_games(username, year, month)

print(f"PGN characters: {len(pgn)}")
print()
print(pgn[:2000])

PGN characters: 173140

[Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.09.27"]
[Round "-"]
[White "brownbambi"]
[Black "tjmoonz98"]
[Result "0-1"]
[CurrentPosition "5k2/5p2/3p1pqp/5K2/5P1P/8/8/8 w - - 3 48"]
[Timezone "UTC"]
[ECO "C44"]
[ECOUrl "https://www.chess.com/openings/Kings-Pawn-Opening-Kings-Knight-Variation-2...Nc6-3.d3"]
[UTCDate "2026.09.27"]
[UTCTime "19:09:59"]
[WhiteElo "475"]
[BlackElo "469"]
[TimeControl "900+10"]
[Termination "tjmoonz98 won by checkmate"]
[StartTime "19:09:59"]
[EndDate "2026.09.27"]
[EndTime "19:35:31"]
[Link "https://www.chess.com/game/live/184469146932"]

1. e4 {[%clk 0:15:06.4]} 1... e5 {[%clk 0:14:56.8]} 2. Nf3 {[%clk 0:15:14.9]} 2... Nc6 {[%clk 0:15:02.7]} 3. d3 {[%clk 0:15:22.3]} 3... Bb4+ {[%clk 0:15:05.7]} 4. c3 {[%clk 0:15:30.3]} 4... Bd6 {[%clk 0:15:10.7]} 5. h3 {[%clk 0:15:39.1]} 5... Nf6 {[%clk 0:15:13.4]} 6. Be2 {[%clk 0:15:45]} 6... O-O {[%clk 0:15:20.9]} 7. O-O {[%clk 0:15:51.3]} 7... b6 {[%clk 0:15:04.4]} 8. d4 {[%clk 0:15:58.3]}

## 3. Parse the first game

A monthly PGN file can contain many games.

For now, we will keep the parser simple and parse just the first game from the PGN data.

In [6]:
# define game and print
game = parse_game(pgn)

print(game)

[Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.09.27"]
[Round "-"]
[White "brownbambi"]
[Black "tjmoonz98"]
[Result "0-1"]
[CurrentPosition "5k2/5p2/3p1pqp/5K2/5P1P/8/8/8 w - - 3 48"]
[Timezone "UTC"]
[ECO "C44"]
[ECOUrl "https://www.chess.com/openings/Kings-Pawn-Opening-Kings-Knight-Variation-2...Nc6-3.d3"]
[UTCDate "2026.09.27"]
[UTCTime "19:09:59"]
[WhiteElo "475"]
[BlackElo "469"]
[TimeControl "900+10"]
[Termination "tjmoonz98 won by checkmate"]
[StartTime "19:09:59"]
[EndDate "2026.09.27"]
[EndTime "19:35:31"]
[Link "https://www.chess.com/game/live/184469146932"]

1. e4 { [%clk 0:15:06.4] } 1... e5 { [%clk 0:14:56.8] } 2. Nf3 { [%clk 0:15:14.9] } 2... Nc6 { [%clk 0:15:02.7] } 3. d3 { [%clk 0:15:22.3] } 3... Bb4+ { [%clk 0:15:05.7] } 4. c3 { [%clk 0:15:30.3] } 4... Bd6 { [%clk 0:15:10.7] } 5. h3 { [%clk 0:15:39.1] } 5... Nf6 { [%clk 0:15:13.4] } 6. Be2 { [%clk 0:15:45] } 6... O-O { [%clk 0:15:20.9] } 7. O-O { [%clk 0:15:51.3] } 7... b6 { [%clk 0:15:04.4] } 8. d4 { [%clk 0:15:5

## 4. Inspect the game headers

PGNs contains metadata about the game in its headers.

Examples include:

- Event
- Site
- Date
- White player
- Black player
- Result
- Opening/ECO information

In [7]:
# print some potentially useful headers
print("White:", game.headers.get("White"))
print("Black:", game.headers.get("Black"))
print("Result:", game.headers.get("Result"))
print("Termination:", game.headers.get("Termination"))
print("Date:", game.headers.get("Date"))
print("Event:", game.headers.get("Event"))
print("ECO:", game.headers.get("ECO"))
print("Number of moves:", game.end().ply())

White: brownbambi
Black: tjmoonz98
Result: 0-1
Termination: tjmoonz98 won by checkmate
Date: 2026.09.27
Event: Live Chess
ECO: C44
Number of moves: 94


## 5. Inspect the moves

`python-chess` represents the moves as a game tree.

We can iterate through the main line to inspect the moves played.

In [8]:
# have to define a board first
board = game.board()

# loop through game object and inspect each move
for move in game.mainline_moves():
    print(board.san(move), end=" ")
    board.push(move)

e4 e5 Nf3 Nc6 d3 Bb4+ c3 Bd6 h3 Nf6 Be2 O-O O-O b6 d4 a5 dxe5 Nxe5 Bf4 Nxf3+ Bxf3 Bxf4 c4 c6 Nc3 Qc7 Na4 Ba6 Re1 Bh2+ Kh1 Bxc4 e5 Bxe5 b3 Bb5 Rb1 Rae8 g4 Bxa4 bxa4 Bc3 Rxe8 Rxe8 g5 Re1+ Qxe1 Bxe1 Rxe1 h6 gxf6 gxf6 Rg1+ Kf8 Be4 b5 axb5 cxb5 Bh7 Qc6+ Kh2 b4 Rg8+ Ke7 Rg7 a4 Bg8 Kf8 Rg4 b3 axb3 axb3 Bh7 Qd6+ Kh1 Qd1+ Kh2 b2 Rb4 Qd6+ Kg2 Qxb4 f4 Qb7+ Kg3 b1=Q Bxb1 Qxb1 Kg4 d6 h4 Qg1+ Kf5 Qg6# 

## 6. Parse Multiple Games

The previous steps were useful for inspecting a game's data output.

But we'll eventually want to parse many games, so let's do so with the `parse_games` function from ChessLab.

In [14]:
# parse many games and print number of games
games = parse_games(pgn)

print(f"Total number of games: {len(games)}")
print("games object is as follows:")
print(games[:3])

Total number of games: 76
games object is as follows:
[<Game at 0x1113a5050 ('brownbambi' vs. 'tjmoonz98', '2026.09.27' at 'Chess.com')>, <Game at 0x110f05f90 ('Dalia-33' vs. 'tjmoonz98', '2026.09.25' at 'Chess.com')>, <Game at 0x11137f8d0 ('tjmoonz98' vs. 'rchrdrchnbch', '2026.09.22' at 'Chess.com')>]


## What we've learned

We have now established the basic pipeline:

`Chess.com API > raw PGN > python-chess > parsed game(s) > game headers and moves`

The next step is to make the parser more useful for ChessLab by extracting
structured information from games, and storing games strategically.

Possible fields include:

- results
- dates
- openings/ECO
- number of moves
- time control
- rated/casual game
- game phases
- individual moves

These structured features will eventually feed into the downstream chess analysis layer.